# Bronze Layer

## Objective

Load the ingested customer support datasets into the Bronze layer while preserving the original data.

Input datasets:
- agent_profiles
- day1_tickets
- day2_tickets

Output:
- bronze_profiles
- bronze_day1
- bronze_day2

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable

CATALOG = "workspace"
SCHEMA = "hrm6131_landing"

# Raw Tables
TBL_PROFILES = f"{CATALOG}.{SCHEMA}.agent_profiles"
TBL_DAY1 = f"{CATALOG}.{SCHEMA}.day1_tickets"
TBL_DAY2 = f"{CATALOG}.{SCHEMA}.day2_tickets"

# Bronze Tables
BRONZE_PROFILES = f"{CATALOG}.{SCHEMA}.bronze_profiles"
BRONZE_DAY1 = f"{CATALOG}.{SCHEMA}.bronze_day1"
BRONZE_DAY2 = f"{CATALOG}.{SCHEMA}.bronze_day2"

print("Configuration loaded.")

Configuration loaded.


## Read Raw Tables

In [0]:
raw_profiles = spark.table(TBL_PROFILES)
raw_day1 = spark.table(TBL_DAY1)
raw_day2 = spark.table(TBL_DAY2)

In [0]:
display(raw_profiles)
display(raw_day1)
display(raw_day2)

agent_id,agent_name,role,team_lead_id
A001,Agent_A001,Junior Support Agent,TL01
A002,Agent_A002,Senior Support Agent,TL01
A003,Agent_A003,Senior Support Agent,TL01
A004,Agent_A004,Junior Support Agent,TL01
A005,Agent_A005,Support Agent,TL01
A006,Agent_A006,Support Agent,TL02
A007,Agent_A007,Junior Support Agent,TL02
A008,Agent_A008,Senior Support Agent,TL02
A009,Agent_A009,Support Agent,TL02
A010,Agent_A010,Junior Support Agent,TL02


ticket_id,agent_id,status,resolution_time,category
TKT00001,A001,Resolved,0h 35m 45s,Technical
TKT00002,A001,Resolved,0h 22m 10s,Billing
TKT00003,A001,Resolved,0h 12m 5s,General
TKT00004,A001,Open,0h 40m 0s,Delivery
TKT00005,A002,Resolved,1h 10m 30s,Returns
TKT00006,A002,Resolved,0h 16m 26s,Billing
TKT00007,A002,Pending,0h 36m 0s,Returns
TKT00008,A003,Resolved,0h 25m 50s,Technical
TKT00009,A003,Resolved,0h 15m 0s,Billing
TKT00010,A003,Resolved,0h 18m 44s,Account


ticket_id,agent_id,status,resolution_time,ticket_category
TKT00200,A001,Resolved,0h 28m 0s,Billing
TKT00201,A001,Resolved,0h 16m 30s,Technical
TKT00202,A002,Resolved,0h 35m 0s,Account
TKT00203,A002,Open,0h 22m 0s,Delivery
TKT00204,A003,Resolved,0h 20m 45s,Returns
TKT00205,A003,Resolved,0h 9m 0s,General
TKT00206,A004,Resolved,0h 31m 0s,Technical
TKT00207,A004,Resolved,0h 18m 15s,Billing
TKT00208,A005,Resolved,0h 26m 0s,Account
TKT00209,A005,Pending,0h 40m 0s,Delivery


## Add Ingestion Metadata

In [0]:
bronze_profiles = (
    raw_profiles
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .withColumn("source_file", F.lit("agent_profiles.csv"))
)

In [0]:
bronze_day1 = (
    raw_day1
    .withColumnRenamed("category", "ticket_category")
    .withColumn("day", F.lit(1))
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .withColumn("source_file", F.lit("day1_tickets.csv"))
)

In [0]:
bronze_day2 = (
    raw_day2
    .withColumn("day", F.lit(2))
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .withColumn("source_file", F.lit("day2_tickets.csv"))
)

## Save Bronze Tables

In [0]:
(
    bronze_profiles.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(BRONZE_PROFILES)
)

(
    bronze_day1.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(BRONZE_DAY1)
)

(
    bronze_day2.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(BRONZE_DAY2)
)

print("Bronze tables created successfully.")

Bronze tables created successfully.


## Validate Bronze Layer

In [0]:
print(f"Bronze Profiles : {spark.table(BRONZE_PROFILES).count()} rows")
print(f"Bronze Day 1    : {spark.table(BRONZE_DAY1).count()} rows")
print(f"Bronze Day 2    : {spark.table(BRONZE_DAY2).count()} rows")

Bronze Profiles : 42 rows
Bronze Day 1    : 123 rows
Bronze Day 2    : 86 rows


In [0]:
display(spark.table(BRONZE_PROFILES))
display(spark.table(BRONZE_DAY1))
display(spark.table(BRONZE_DAY2))

agent_id,agent_name,role,team_lead_id,ingestion_timestamp,source_file
A001,Agent_A001,Junior Support Agent,TL01,2026-07-22T05:51:06.716Z,agent_profiles.csv
A002,Agent_A002,Senior Support Agent,TL01,2026-07-22T05:51:06.716Z,agent_profiles.csv
A003,Agent_A003,Senior Support Agent,TL01,2026-07-22T05:51:06.716Z,agent_profiles.csv
A004,Agent_A004,Junior Support Agent,TL01,2026-07-22T05:51:06.716Z,agent_profiles.csv
A005,Agent_A005,Support Agent,TL01,2026-07-22T05:51:06.716Z,agent_profiles.csv
A006,Agent_A006,Support Agent,TL02,2026-07-22T05:51:06.716Z,agent_profiles.csv
A007,Agent_A007,Junior Support Agent,TL02,2026-07-22T05:51:06.716Z,agent_profiles.csv
A008,Agent_A008,Senior Support Agent,TL02,2026-07-22T05:51:06.716Z,agent_profiles.csv
A009,Agent_A009,Support Agent,TL02,2026-07-22T05:51:06.716Z,agent_profiles.csv
A010,Agent_A010,Junior Support Agent,TL02,2026-07-22T05:51:06.716Z,agent_profiles.csv


ticket_id,agent_id,status,resolution_time,ticket_category,day,ingestion_timestamp,source_file
TKT00001,A001,Resolved,0h 35m 45s,Technical,1,2026-07-22T05:51:13.756Z,day1_tickets.csv
TKT00002,A001,Resolved,0h 22m 10s,Billing,1,2026-07-22T05:51:13.756Z,day1_tickets.csv
TKT00003,A001,Resolved,0h 12m 5s,General,1,2026-07-22T05:51:13.756Z,day1_tickets.csv
TKT00004,A001,Open,0h 40m 0s,Delivery,1,2026-07-22T05:51:13.756Z,day1_tickets.csv
TKT00005,A002,Resolved,1h 10m 30s,Returns,1,2026-07-22T05:51:13.756Z,day1_tickets.csv
TKT00006,A002,Resolved,0h 16m 26s,Billing,1,2026-07-22T05:51:13.756Z,day1_tickets.csv
TKT00007,A002,Pending,0h 36m 0s,Returns,1,2026-07-22T05:51:13.756Z,day1_tickets.csv
TKT00008,A003,Resolved,0h 25m 50s,Technical,1,2026-07-22T05:51:13.756Z,day1_tickets.csv
TKT00009,A003,Resolved,0h 15m 0s,Billing,1,2026-07-22T05:51:13.756Z,day1_tickets.csv
TKT00010,A003,Resolved,0h 18m 44s,Account,1,2026-07-22T05:51:13.756Z,day1_tickets.csv


ticket_id,agent_id,status,resolution_time,ticket_category,day,ingestion_timestamp,source_file
TKT00200,A001,Resolved,0h 28m 0s,Billing,2,2026-07-22T05:51:18.066Z,day2_tickets.csv
TKT00201,A001,Resolved,0h 16m 30s,Technical,2,2026-07-22T05:51:18.066Z,day2_tickets.csv
TKT00202,A002,Resolved,0h 35m 0s,Account,2,2026-07-22T05:51:18.066Z,day2_tickets.csv
TKT00203,A002,Open,0h 22m 0s,Delivery,2,2026-07-22T05:51:18.066Z,day2_tickets.csv
TKT00204,A003,Resolved,0h 20m 45s,Returns,2,2026-07-22T05:51:18.066Z,day2_tickets.csv
TKT00205,A003,Resolved,0h 9m 0s,General,2,2026-07-22T05:51:18.066Z,day2_tickets.csv
TKT00206,A004,Resolved,0h 31m 0s,Technical,2,2026-07-22T05:51:18.066Z,day2_tickets.csv
TKT00207,A004,Resolved,0h 18m 15s,Billing,2,2026-07-22T05:51:18.066Z,day2_tickets.csv
TKT00208,A005,Resolved,0h 26m 0s,Account,2,2026-07-22T05:51:18.066Z,day2_tickets.csv
TKT00209,A005,Pending,0h 40m 0s,Delivery,2,2026-07-22T05:51:18.066Z,day2_tickets.csv
